# Effect of short closed loops on local-geometry statistics

This notebook compares the three local-geometry marginals used in Figure 5 at the lowest and highest salt concentrations. The three estimators are: (1) the direct Kac--Rice/coarea random-wave calculation, (2) explicit contour tracing, and (3) the same trace after removing only genuine closed loops shorter than a configurable cutoff.

The explicit tracer does **not** impose periodic wrapping. The gamma-radial wave vectors are continuous rather than integer-valued, so the sampled field is not periodic across opposite box faces. Curves that reach a box face are retained as boundary-censored open contours; any graph cycle touching a face is cut at that face and is not counted as a loop. Internal degree-4 and degree-6 marching ambiguities are continued by the smoothest opposite-direction pairing, while a loop is declared closed only by graph topology. Rare odd-degree breaks are retained as topology-censored open segments after endpoint trimming and are never classified as loops.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import least_squares
from scipy.special import gammaln, roots_genlaguerre
from scipy.stats import t as student_t

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'smpl' / 'curvefit' / 'cf_ca_interaction.py').exists():
    raise RuntimeError('Run this notebook from the project_randomcxl root directory')
CURVEFIT_DIR = PROJECT_ROOT / 'smpl' / 'curvefit'
if str(CURVEFIT_DIR) not in sys.path:
    sys.path.insert(0, str(CURVEFIT_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cf_ca_interaction as geometry
import rw_line_network as rwn

QUICK_CHECK = os.environ.get('LOOP_NOTEBOOK_QUICK', '0') == '1'
FIT_DIR = CURVEFIT_DIR / 'output' / 'ca_aniso' / 'gamma_OH3_best'
FIT_PATTERN = 'ca_*mM_gamma_radial_unguided_gamma_radial_full_geom_OH3_fit_parameters.csv'
DEFAULT_OUTPUT_DIR = FIT_DIR / 'comparison' / 'gamma_radial_full_geom' / 'geometry' / 'comparison' / 'loop_test'
OUTPUT_DIR = Path(os.environ.get('LOOP_NOTEBOOK_OUTPUT_DIR', DEFAULT_OUTPUT_DIR)).resolve()
CACHE_DIR = OUTPUT_DIR / 'trace_cache'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Production controls. LOOP_NOTEBOOK_QUICK=1 selects a small smoke-test workload.
DIRECT_SAMPLE_POWER = 12 if QUICK_CHECK else 17
DIRECT_SEEDS = (73129, 73130) if QUICK_CHECK else (73129, 73130, 73131, 73132)
TRACE_SEEDS = (894894,) if QUICK_CHECK else (894894, 894895, 894896, 894897, 894898, 894899)
TRACE_GRID_SIZE = 40 if QUICK_CHECK else 96
TRACE_RADIAL_NODES = 4
TRACE_POLAR_NODES = 4
TRACE_AZIMUTHAL_NODES = 8
TRACE_NUM_MODES = TRACE_RADIAL_NODES * TRACE_POLAR_NODES * TRACE_AZIMUTHAL_NODES
TRACE_K0 = 3.5 if QUICK_CHECK else 8.0
TRACE_Q_SPACING = 0.16 if QUICK_CHECK else 0.08
BOUNDARY_TOL_GRID = 0.55
ENDPOINT_TRIM_Q = 1.0
PROJECTION_RESIDUAL_TOL = 1e-8
MAX_MEAN_QUALITY_REJECTED_ARCLENGTH = 0.02
MAX_SINGLE_QUALITY_REJECTED_ARCLENGTH = 0.05
SHORT_LOOP_CUTOFF_WAVELENGTHS = 3.0
RECOMPUTE_TRACE_CACHE = False
ANISOTROPY_AXIS = 2

HISTOGRAM_BINS = {'kappa': 100, 'kappa_tau': 100, 'kappa_prime': 100}
HISTOGRAM_RANGES = {'kappa': (0.0, 5.0), 'kappa_tau': (-6.0, 6.0), 'kappa_prime': (-16.0, 16.0)}
DISPLAY_TICKS = {'kappa': (0, 1, 2, 3, 4, 5), 'kappa_tau': (-6, -3, 0, 3, 6), 'kappa_prime': (-15, -10, -5, 0, 5, 10, 15)}
VARIABLE_SPECS = (
    ('kappa', 0, r'$\kappa/k_{\mathrm{rms},0}$', r'$p(\kappa)$'),
    ('kappa_tau', 2, r'$\kappa\tau/k_{\mathrm{rms},0}^2$', r'$p(\kappa\tau)$'),
    ('kappa_prime', 1, r"$\kappa'/k_{\mathrm{rms},0}^2$", r"$p(\kappa')$"),
)

all_spectra = geometry.discover_fit_spectra(FIT_DIR, FIT_PATTERN)
selected_spectra = (min(all_spectra, key=lambda row: row.concentration_mM),
                    max(all_spectra, key=lambda row: row.concentration_mM))
reference_spectrum = geometry.select_reference_spectrum(all_spectra, 'zero')
K_EFF_0 = reference_spectrum.k_eff
GEOMETRY_SCALES = np.array([K_EFF_0, K_EFF_0**2, K_EFF_0**2])
SHORT_LOOP_CUTOFF_QREF = SHORT_LOOP_CUTOFF_WAVELENGTHS * 2.0 * np.pi
hencky_all_by_salt = geometry.fitted_hencky_tensors(all_spectra, axis=ANISOTROPY_AXIS)
hencky_by_salt = {row.concentration_mM: hencky_all_by_salt[row.concentration_mM]
                  for row in selected_spectra}

print('salt cases:', [row.concentration_mM for row in selected_spectra], 'mM')
print(f'k_eff,0 = {K_EFF_0:.8f} A^-1')
print(f'short-loop cutoff: L*k_eff,0 < {SHORT_LOOP_CUTOFF_QREF:.5f} '      f'({SHORT_LOOP_CUTOFF_WAVELENGTHS:g} wavelengths)')
print('quick check:', QUICK_CHECK)


## Direct random-wave reference

The direct calculation samples the conditional derivative distribution at a random line point and uses its coarea weight. Independent scrambled Sobol replicates quantify integration error. The same fitted affine stretch used for Figure 5 is applied before evaluating the local geometry.


In [ ]:
direct_replicates: dict[float, list[dict[str, np.ndarray]]] = {
    row.concentration_mM: [] for row in selected_spectra
}
direct_diagnostics = []
for seed in DIRECT_SEEDS:
    sampled = geometry.sample_local_geometry_series(
        selected_spectra, sample_power=DIRECT_SAMPLE_POWER, seed=seed,
        aniso=True, H_by_concentration=hencky_by_salt,
        alignment_axis=ANISOTROPY_AXIS,
    )
    for row in selected_spectra:
        sample = sampled[row.concentration_mM]
        direct_replicates[row.concentration_mM].append({
            'state': sample.state / GEOMETRY_SCALES,
            'weights': sample.weights,
        })
        direct_diagnostics.append({
            'concentration_mM': row.concentration_mM, 'seed': seed,
            'sample_count': len(sample.weights),
            'stable_fraction': sample.stable_fraction,
            'effective_sample_size': 1.0 / np.sum(sample.weights**2),
        })
direct_diagnostics


## Boundary-aware explicit tracing

The segment graph is continued through every even-degree internal ambiguity by minimizing the total turning cost. A graph cycle is closed explicitly by its final graph edge. Open traces ending on the box faces are boundary-censored; rare odd-degree or one-sided breaks are topology-censored. Both types remain useful for local statistics after trimming both endpoints, but neither is eligible for loop-size exclusion. Derivatives through third order are evaluated analytically from the same Fourier realization after safeguarded Newton projection onto both zero surfaces.


In [ ]:
def quadrature_wave_coefficients(mean_k, r_sigma_k, rng):
    """Spectral quadrature exact for isotropic derivative moments through order six."""
    shape = 1.0 / r_sigma_k**2
    scale = mean_k * r_sigma_k**2
    radial_nodes, radial_weights = roots_genlaguerre(
        TRACE_RADIAL_NODES, shape - 1.0)
    radii = scale * radial_nodes
    radial_weights = radial_weights / np.exp(gammaln(shape))

    z_nodes, z_weights = np.polynomial.legendre.leggauss(TRACE_POLAR_NODES)
    azimuth = 2.0 * np.pi * (np.arange(TRACE_AZIMUTHAL_NODES) + 0.5) / TRACE_AZIMUTHAL_NODES
    directions = []
    angular_weights = []
    for z, z_weight in zip(z_nodes, z_weights):
        radial_xy = np.sqrt(max(0.0, 1.0 - z**2))
        for phi in azimuth:
            directions.append((radial_xy * np.cos(phi), radial_xy * np.sin(phi), z))
            angular_weights.append(0.5 * z_weight / TRACE_AZIMUTHAL_NODES)
    directions = np.asarray(directions)
    angular_weights = np.asarray(angular_weights)

    rotation, triangular = np.linalg.qr(rng.normal(size=(3, 3)))
    rotation *= np.sign(np.diag(triangular))[None, :]
    if np.linalg.det(rotation) < 0:
        rotation[:, 0] *= -1.0
    directions = directions @ rotation.T
    vectors = (radii[:, None, None] * directions[None, :, :]).reshape(-1, 3)
    probability_weights = (radial_weights[:, None] * angular_weights[None, :]).ravel()
    amplitudes = np.sqrt(probability_weights)
    phases = rng.uniform(0.0, 2.0 * np.pi, size=len(vectors))
    return rwn.WaveCoefficients(vectors, amplitudes, phases)


def continued_segment_paths(poly):
    """Return smoothly continued paths and graph-exact closure flags."""
    if poly.n_points == 0 or poly.n_lines == 0:
        return ()
    raw_points = np.asarray(poly.points, dtype=float)
    lines = np.asarray(poly.lines, dtype=np.int64).reshape(-1, 3)
    if np.any(lines[:, 0] != 2):
        raise ValueError('expected two-point line cells')
    points, inverse = np.unique(raw_points, axis=0, return_inverse=True)
    edges = np.sort(inverse[lines[:, 1:]], axis=1)
    edges = np.unique(edges[edges[:, 0] != edges[:, 1]], axis=0)
    adjacency = [[] for _ in range(len(points))]
    for first, second in edges:
        adjacency[first].append(int(second))
        adjacency[second].append(int(first))

    continuation = {}
    def smooth_pairs(node, neighbors):
        direction = points[neighbors] - points[node]
        direction /= np.linalg.norm(direction, axis=1)[:, None]
        cost = direction @ direction.T
        def best(indices):
            if not indices:
                return 0.0, []
            first = indices[0]
            candidates = []
            for offset in range(1, len(indices)):
                second = indices[offset]
                rest = indices[1:offset] + indices[offset + 1:]
                following_cost, following_pairs = best(rest)
                candidates.append((cost[first, second] + following_cost,
                                   [(neighbors[first], neighbors[second])] + following_pairs))
            return min(candidates, key=lambda item: item[0])
        return best(tuple(range(len(neighbors))))[1]

    for node, neighbors in enumerate(adjacency):
        if len(neighbors) == 2:
            pairs = [(neighbors[0], neighbors[1])]
        elif len(neighbors) > 2 and len(neighbors) % 2 == 0:
            pairs = smooth_pairs(node, neighbors)
        else:
            pairs = []
        for first, second in pairs:
            continuation[node, first] = second
            continuation[node, second] = first

    visited = set()
    def walk(first, second):
        path = [first, second]
        visited.add((min(first, second), max(first, second)))
        previous, current = first, second
        closed = False
        while True:
            following = continuation.get((current, previous))
            if following is None:
                break
            edge = (min(current, following), max(current, following))
            if following == path[0]:
                if edge not in visited:
                    visited.add(edge)
                path.append(following)
                closed = True
                break
            if edge in visited:
                break
            visited.add(edge)
            path.append(following)
            previous, current = current, following
        return np.asarray(points[path], dtype=float), closed

    paths = []
    for node, neighbors in enumerate(adjacency):
        if neighbors and all((node, neighbor) in continuation for neighbor in neighbors):
            continue
        for neighbor in neighbors:
            edge = (min(node, neighbor), max(node, neighbor))
            if edge not in visited:
                paths.append(walk(node, neighbor))
    for first, second in edges:
        edge = (int(first), int(second))
        if edge not in visited:
            paths.append(walk(*edge))
    return tuple((path, closed) for path, closed in paths if len(path) >= 2)


def trace_field_local_geometry(spectrum, H, config):
    rwn.GRID_SIZE = config.grid_size
    rwn.NUM_BLOCK = config.num_blocks
    rwn.BLOCK_OVERLAP = config.block_overlap
    rwn.RANDOM_SEED = config.random_seed
    rwn.NUM_MODES = (config.num_modes,) * 3
    rwn.K_DISTRIBUTION = 'gamma_radial'
    rwn.K0 = (config.trace_k0,) * 3
    rwn.r_SIGMA_K = (spectrum.r_sigma_k,) * 3
    rwn.SHARED_K_VECTORS = False
    rwn.COUPLE_PHI2_PHI3 = False
    rwn.USE_VORTEX_TRACING = True
    rwn.VORTEX_FACE_PREFILTER = True
    rwn.SMOOTH_VORTEX_LINES = False

    rng = np.random.default_rng(config.random_seed)
    coefficients1 = quadrature_wave_coefficients(
        config.trace_k0, spectrum.r_sigma_k, rng)
    coefficients2 = quadrature_wave_coefficients(
        config.trace_k0, spectrum.r_sigma_k, rng)
    def weighted_radial_moment(coefficients, order):
        weights = coefficients.amplitudes**2
        radii = np.linalg.norm(coefficients.k_vectors, axis=1)
        return float(np.dot(weights, radii**order) / np.sum(weights))
    realized_moments = {order: 0.5 * (
        weighted_radial_moment(coefficients1, order)
        + weighted_radial_moment(coefficients2, order)) for order in (2, 4, 6)}
    trace_k_eff = (2.0 * np.pi / config.grid_size) * np.sqrt(realized_moments[2])
    phi1 = rwn.build_random_wave_field_from_coefficients(config.grid_size, coefficients1)
    phi2 = rwn.build_random_wave_field_from_coefficients(config.grid_size, coefficients2)

    def wave_jet(points, coefficients, order=0):
        q = (2.0 * np.pi / config.grid_size) * coefficients.k_vectors
        phase = points @ q.T + coefficients.phases
        cw = np.cos(phase) * coefficients.amplitudes
        sw = np.sin(phase) * coefficients.amplitudes
        value = np.sum(cw, axis=1)
        if order == 0:
            return (value,)
        gradient = (-sw) @ q
        if order == 1:
            return value, gradient
        hessian = np.einsum('nm,mi,mj->nij', -cw, q, q, optimize=True)
        if order == 2:
            return value, gradient, hessian
        third = np.einsum('nm,mi,mj,mk->nijk', sw, q, q, q, optimize=True)
        return value, gradient, hessian, third

    def plane_coefficients(g1, g2, rhs1, rhs2):
        a11 = np.einsum('ij,ij->i', g1, g1)
        a12 = np.einsum('ij,ij->i', g1, g2)
        a22 = np.einsum('ij,ij->i', g2, g2)
        determinant = a11 * a22 - a12 * a12
        safe = np.abs(determinant) > 1e-14 * np.maximum(a11 * a22, 1.0)
        c1 = np.zeros_like(rhs1)
        c2 = np.zeros_like(rhs2)
        c1[safe] = (rhs1[safe] * a22[safe] - rhs2[safe] * a12[safe]) / determinant[safe]
        c2[safe] = (rhs2[safe] * a11[safe] - rhs1[safe] * a12[safe]) / determinant[safe]
        return c1, c2, safe

    def project(points, iterations=4):
        output = np.asarray(points, dtype=float).copy()
        for _ in range(iterations):
            f1, g1 = wave_jet(output, coefficients1, 1)
            f2, g2 = wave_jet(output, coefficients2, 1)
            c1, c2, safe = plane_coefficients(g1, g2, -f1, -f2)
            step = c1[:, None] * g1 + c2[:, None] * g2
            step_norm = np.linalg.norm(step, axis=1)
            step *= np.minimum(1.0, 0.75 / np.maximum(step_norm, 1e-15))[:, None]
            current_norm = np.sqrt(f1**2 + f2**2)
            accepted = ~safe
            factor = np.ones(len(output))
            for _ in range(8):
                pending = ~accepted
                if not np.any(pending):
                    break
                trial = output[pending] + factor[pending, None] * step[pending]
                trial_f1 = wave_jet(trial, coefficients1, 0)[0]
                trial_f2 = wave_jet(trial, coefficients2, 0)[0]
                improved_local = np.sqrt(trial_f1**2 + trial_f2**2) < current_norm[pending]
                pending_indices = np.flatnonzero(pending)
                improved = pending_indices[improved_local]
                output[improved] = trial[improved_local]
                accepted[improved] = True
                factor[pending_indices[~improved_local]] *= 0.5
        return output

    def analytic_geometry(points):
        tangents = np.empty_like(points)
        r2out = np.empty_like(points)
        r3out = np.empty_like(points)
        for start in range(0, len(points), 4096):
            stop = min(start + 4096, len(points))
            local = points[start:stop]
            _, g1, h1, d31 = wave_jet(local, coefficients1, 3)
            _, g2, h2, d32 = wave_jet(local, coefficients2, 3)
            tangent = np.cross(g1, g2)
            tangent /= np.linalg.norm(tangent, axis=1)[:, None]
            matrix = np.stack((g1, g2, tangent), axis=1)
            rhs2 = np.column_stack((-geometry.contract2(h1, tangent, tangent),
                                    -geometry.contract2(h2, tangent, tangent),
                                    np.zeros(len(local))))
            local_r2 = np.linalg.solve(matrix, rhs2[..., None])[..., 0]
            rhs3 = np.column_stack((
                -geometry.contract3(d31, tangent, tangent, tangent)
                - 3.0 * geometry.contract2(h1, tangent, local_r2),
                -geometry.contract3(d32, tangent, tangent, tangent)
                - 3.0 * geometry.contract2(h2, tangent, local_r2),
                -np.einsum('ni,ni->n', local_r2, local_r2),
            ))
            local_r3 = np.linalg.solve(matrix, rhs3[..., None])[..., 0]
            tangents[start:stop] = tangent
            r2out[start:stop] = local_r2
            r3out[start:stop] = local_r3
        return tangents, r2out, r3out

    segment_polydata = rwn.trace_vortex_segments(phi1, phi2)
    raw_paths = continued_segment_paths(segment_polydata)
    spacing_grid = config.q_spacing / trace_k_eff
    length_scale = trace_k_eff / spectrum.k_eff
    boundary_high = config.grid_size - 1.0 - BOUNDARY_TOL_GRID
    def on_boundary(point):
        return bool(np.any(point <= BOUNDARY_TOL_GRID) or np.any(point >= boundary_high))

    states = []
    contour_points = []
    contour_tangents = []
    contour_ids = []
    contour_closed = []
    contour_boundary = []
    contour_topology_censored = []
    contour_lengths = []
    residuals = []
    spacing_ratios = []
    turn_angles = []
    rejected_short = rejected_topology = rejected_singular = rejected_projection = 0
    raw_arclength_grid = rejected_quality_arclength_grid = rejected_short_arclength_grid = 0.0

    for raw_path, graph_closed in raw_paths:
        raw_length_grid = float(np.sum(np.linalg.norm(np.diff(raw_path, axis=0), axis=1)))
        raw_arclength_grid += raw_length_grid
        touches_boundary = np.array([on_boundary(point) for point in raw_path])
        if graph_closed and np.any(touches_boundary):
            cycle = raw_path[:-1]
            cut = int(np.flatnonzero(touches_boundary[:-1])[0])
            raw_path = np.vstack((cycle[cut:], cycle[:cut + 1]))
            graph_closed = False
            touches_boundary = np.array([on_boundary(point) for point in raw_path])
        boundary_censored = (not graph_closed and on_boundary(raw_path[0])
                             and on_boundary(raw_path[-1]))
        topology_censored = not graph_closed and not boundary_censored
        valid_closed = graph_closed and not np.any(touches_boundary)
        if graph_closed and not valid_closed:
            rejected_topology += 1
            rejected_quality_arclength_grid += raw_length_grid
            continue
        try:
            projected_path = project(raw_path, iterations=8)
            sampled = geometry.resample_polyline_by_arclength(projected_path, spacing_grid)
        except ValueError:
            rejected_short += 1
            rejected_short_arclength_grid += raw_length_grid
            continue
        sampled = project(sampled, iterations=10)
        f1 = wave_jet(sampled, coefficients1, 0)[0] / rwn.coefficient_field_std(coefficients1)
        f2 = wave_jet(sampled, coefficients2, 0)[0] / rwn.coefficient_field_std(coefficients2)
        local_residual = np.sqrt(f1**2 + f2**2)
        for bad_index in np.flatnonzero(local_residual > PROJECTION_RESIDUAL_TOL):
            def residual_function(point):
                value1 = wave_jet(point[None, :], coefficients1, 0)[0][0]
                value2 = wave_jet(point[None, :], coefficients2, 0)[0][0]
                return np.array([value1, value2])
            def residual_jacobian(point):
                gradient1 = wave_jet(point[None, :], coefficients1, 1)[1][0]
                gradient2 = wave_jet(point[None, :], coefficients2, 1)[1][0]
                return np.vstack((gradient1, gradient2))
            refined = least_squares(
                residual_function, sampled[bad_index], jac=residual_jacobian,
                method='trf', ftol=1e-13, xtol=1e-13, gtol=1e-13, max_nfev=100,
            )
            sampled[bad_index] = refined.x
        f1 = wave_jet(sampled, coefficients1, 0)[0] / rwn.coefficient_field_std(coefficients1)
        f2 = wave_jet(sampled, coefficients2, 0)[0] / rwn.coefficient_field_std(coefficients2)
        local_residual = np.sqrt(f1**2 + f2**2)
        if np.max(local_residual) > PROJECTION_RESIDUAL_TOL:
            rejected_projection += 1
            rejected_quality_arclength_grid += raw_length_grid
            continue
        try:
            tangent, r2, r3 = analytic_geometry(sampled)
        except (ValueError, np.linalg.LinAlgError, FloatingPointError):
            rejected_singular += 1
            rejected_quality_arclength_grid += raw_length_grid
            continue
        if not all(np.all(np.isfinite(array)) for array in (sampled, tangent, r2, r3)):
            rejected_singular += 1
            rejected_quality_arclength_grid += raw_length_grid
            continue

        residuals.append(local_residual)
        physical_points = sampled * length_scale
        physical_r2 = r2 / length_scale
        physical_r3 = r3 / length_scale**2
        if valid_closed:
            physical_points = np.vstack((physical_points, physical_points[0]))
            tangent = np.vstack((tangent, tangent[0]))
            physical_r2 = np.vstack((physical_r2, physical_r2[0]))
            physical_r3 = np.vstack((physical_r3, physical_r3[0]))
        source_trace = geometry.ContourTrace(
            (physical_points,), (tangent,), (physical_r2,), (physical_r3,),
            spectrum.k_eff, config.q_spacing,
            {'closed': valid_closed, 'boundary_censored': boundary_censored,
             'topology_censored': topology_censored},
        )
        try:
            transformed = geometry.transform_contour_trace(source_trace, H)
        except RuntimeError:
            rejected_short += 1
            rejected_short_arclength_grid += raw_length_grid
            continue
        points = transformed.points[0]
        local_tangent = transformed.tangents[0]
        local_state = geometry.trace_local_state(transformed, normalize=False)[0]
        if not valid_closed:
            trim = int(np.ceil(ENDPOINT_TRIM_Q / transformed.q_spacing))
            if len(points) <= 2 * trim + 3:
                rejected_short += 1
                rejected_short_arclength_grid += raw_length_grid
                continue
            points = points[trim:-trim]
            local_tangent = local_tangent[trim:-trim]
            local_state = local_state[trim:-trim]
        segments = np.linalg.norm(np.diff(points, axis=0), axis=1)
        contour_length = float(np.sum(segments))
        if valid_closed:
            contour_length += float(np.linalg.norm(points[-1] - points[0]))
        if len(segments):
            spacing_ratios.append(segments / transformed.physical_spacing)
        if len(local_tangent) > 1:
            dots = np.einsum('ni,ni->n', local_tangent[:-1], local_tangent[1:])
            turn_angles.append(np.degrees(np.arccos(np.clip(np.abs(dots), 0.0, 1.0))))
        contour_id = len(contour_closed)
        states.append(local_state)
        contour_points.append(points)
        contour_tangents.append(local_tangent)
        contour_ids.append(np.full(len(local_state), contour_id, dtype=np.int64))
        contour_closed.append(valid_closed)
        contour_boundary.append(boundary_censored)
        contour_topology_censored.append(topology_censored)
        contour_lengths.append(contour_length)

    if not states:
        raise RuntimeError('No valid contours survived tracing and boundary checks')
    residual = np.concatenate(residuals) if residuals else np.array([np.nan])
    spacing_ratio = np.concatenate(spacing_ratios) if spacing_ratios else np.array([np.nan])
    turn_angle = np.concatenate(turn_angles) if turn_angles else np.array([np.nan])
    contour_lengths = np.asarray(contour_lengths, dtype=float)
    contour_closed = np.asarray(contour_closed, dtype=bool)
    contour_boundary = np.asarray(contour_boundary, dtype=bool)
    contour_topology_censored = np.asarray(contour_topology_censored, dtype=bool)
    contour_id = np.concatenate(contour_ids)
    short_loop = contour_closed & (contour_lengths * K_EFF_0 < SHORT_LOOP_CUTOFF_QREF)
    state = np.vstack(states) / GEOMETRY_SCALES
    keep_without_short_loops = ~short_loop[contour_id]
    diagnostics = {
        'raw_path_count': len(raw_paths),
        'retained_contour_count': len(contour_lengths),
        'closed_loop_count': int(np.sum(contour_closed)),
        'boundary_censored_count': int(np.sum(contour_boundary)),
        'topology_censored_count': int(np.sum(contour_topology_censored)),
        'short_loop_count': int(np.sum(short_loop)),
        'rejected_topology_count': rejected_topology,
        'rejected_too_short_count': rejected_short,
        'rejected_singular_count': rejected_singular,
        'rejected_projection_count': rejected_projection,
        'quality_rejection_path_fraction': float(
            (rejected_topology + rejected_singular + rejected_projection) / max(len(raw_paths), 1)),
        'quality_rejected_raw_arclength_fraction': float(
            rejected_quality_arclength_grid / max(raw_arclength_grid, np.finfo(float).tiny)),
        'too_short_raw_arclength_fraction': float(
            rejected_short_arclength_grid / max(raw_arclength_grid, np.finfo(float).tiny)),
        'point_count': len(state),
        'point_count_without_short_loops': int(np.sum(keep_without_short_loops)),
        'total_arclength_A': float(np.sum(contour_lengths)),
        'short_loop_arclength_fraction': float(
            np.sum(contour_lengths[short_loop]) / np.sum(contour_lengths)),
        'topology_censored_arclength_fraction': float(
            np.sum(contour_lengths[contour_topology_censored]) / np.sum(contour_lengths)),
        'zero_surface_residual_rms': float(np.sqrt(np.mean(residual**2))),
        'zero_surface_residual_max': float(np.max(residual)),
        'spacing_ratio_mean': float(np.mean(spacing_ratio)),
        'spacing_ratio_std': float(np.std(spacing_ratio)),
        'turn_angle_p99_deg': float(np.quantile(turn_angle, 0.99)),
        'raw_trace_k_eff_grid^-1': float(trace_k_eff),
        **{f'spectral_moment_{order}_relative_error': float(
            realized_moments[order] / geometry.gamma_radial_moment(
                order, spectrum.r_sigma_k, config.trace_k0 * np.sqrt(1.0 + spectrum.r_sigma_k**2)) - 1.0)
           for order in (2, 4, 6)},
    }
    return {
        'state': state,
        'contour_points_A': tuple(contour_points),
        'contour_tangents': tuple(contour_tangents),
        'keep_without_short_loops': keep_without_short_loops,
        'contour_id': contour_id,
        'contour_closed': contour_closed,
        'contour_boundary': contour_boundary,
        'contour_topology_censored': contour_topology_censored,
        'contour_length_A': contour_lengths,
        'short_loop': short_loop,
        'diagnostics': diagnostics,
    }


def trace_cache_path(spectrum, H, config):
    signature = {
        'schema': 10, 'spectrum': [spectrum.k_eff, spectrum.r_sigma_k],
        'H': np.asarray(H).round(14).tolist(), 'config': asdict(config),
        'boundary_tol_grid': BOUNDARY_TOL_GRID, 'endpoint_trim_q': ENDPOINT_TRIM_Q,
        'projection_residual_tol': PROJECTION_RESIDUAL_TOL,
        'spectral_quadrature': [TRACE_RADIAL_NODES, TRACE_POLAR_NODES,
                                TRACE_AZIMUTHAL_NODES],
        'k_eff_0': K_EFF_0, 'short_cutoff_qref': SHORT_LOOP_CUTOFF_QREF,
    }
    digest = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:12]
    return CACHE_DIR / f'ca_{spectrum.concentration_mM:g}mM_seed_{config.random_seed}_{digest}.npz'


def save_trace_result(path, result):
    names = np.asarray(list(result['diagnostics']))
    values = np.asarray([result['diagnostics'][name] for name in names], dtype=float)
    packed_points, contour_offsets = geometry.packed_cells(result['contour_points_A'])
    packed_tangents, _ = geometry.packed_cells(result['contour_tangents'])
    np.savez_compressed(
        path, state=result['state'], keep_without_short_loops=result['keep_without_short_loops'],
        contour_id=result['contour_id'], contour_closed=result['contour_closed'],
        contour_boundary=result['contour_boundary'], contour_length_A=result['contour_length_A'],
        contour_topology_censored=result['contour_topology_censored'],
        short_loop=result['short_loop'], contour_points_A=packed_points,
        contour_tangents=packed_tangents, contour_offsets=contour_offsets,
        diagnostic_names=names, diagnostic_values=values,
    )


def load_trace_result(path):
    with np.load(path, allow_pickle=False) as data:
        names = data['diagnostic_names'].astype(str)
        values = data['diagnostic_values']
        return {
            'state': data['state'],
            'contour_points_A': geometry.unpacked_cells(
                data['contour_points_A'], data['contour_offsets']),
            'contour_tangents': geometry.unpacked_cells(
                data['contour_tangents'], data['contour_offsets']),
            'keep_without_short_loops': data['keep_without_short_loops'].astype(bool),
            'contour_id': data['contour_id'],
            'contour_closed': data['contour_closed'].astype(bool),
            'contour_boundary': data['contour_boundary'].astype(bool),
            'contour_topology_censored': data['contour_topology_censored'].astype(bool),
            'contour_length_A': data['contour_length_A'],
            'short_loop': data['short_loop'].astype(bool),
            'diagnostics': dict(zip(names, values)),
        }


In [ ]:
trace_replicates: dict[float, list[dict[str, np.ndarray]]] = {
    row.concentration_mM: [] for row in all_spectra
}
trace_diagnostic_rows = []


def ensure_trace_cache(row, seed):
    concentration = row.concentration_mM
    config = geometry.TraceConfig(
        grid_size=TRACE_GRID_SIZE, num_blocks=1, block_overlap=0,
        num_modes=TRACE_NUM_MODES, trace_k0=TRACE_K0,
        random_seed=seed, seed_spacing=0.35, q_spacing=TRACE_Q_SPACING,
    )
    cache_path = trace_cache_path(row, hencky_all_by_salt[concentration], config)
    if not cache_path.exists() or RECOMPUTE_TRACE_CACHE:
        print(f'[trace] {concentration:g} mM, seed {seed}')
        result = trace_field_local_geometry(
            row, hencky_all_by_salt[concentration], config)
        save_trace_result(cache_path, result)
    return concentration, seed, cache_path


trace_jobs = [(row, seed) for row in all_spectra for seed in TRACE_SEEDS]
# VTK contour extraction is not thread-safe across independent fields; keep
# production tracing serial. Cached traces make subsequent executions fast.
trace_workers = 1
with ThreadPoolExecutor(max_workers=trace_workers) as executor:
    futures = [executor.submit(ensure_trace_cache, row, seed)
               for row, seed in trace_jobs]
    cache_records = [future.result() for future in futures]

for concentration, seed, cache_path in cache_records:
    result = load_trace_result(cache_path)
    trace_replicates[concentration].append(result)
    trace_diagnostic_rows.append({
        'concentration_mM': concentration, 'seed': seed,
        **result['diagnostics'],
    })

trace_diagnostic_rows


## Marginal distributions and uncertainty

All three curves use the fixed Figure 5 bins and ranges. Probability outside the displayed range is not silently renormalized. Each line is the pooled estimate. Replicate intervals are calculated for numerical diagnostics but are not drawn on the figure.


In [ ]:
def histogram_density(values, variable, weights=None):
    mass, edges = np.histogram(
        values, bins=HISTOGRAM_BINS[variable], range=HISTOGRAM_RANGES[variable],
        weights=weights, density=False,
    )
    total = len(values) if weights is None else float(np.sum(weights))
    density = mass / total / np.diff(edges)
    return 0.5 * (edges[:-1] + edges[1:]), density, float(np.sum(mass) / total)


def replicate_interval(densities, confidence=0.95):
    values = np.asarray(densities, dtype=float)
    center = np.mean(values, axis=0)
    if len(values) < 2:
        return center, center
    critical = student_t.ppf(0.5 + confidence / 2.0, df=len(values) - 1)
    half_width = critical * np.std(values, axis=0, ddof=1) / np.sqrt(len(values))
    return np.clip(center - half_width, 0.0, None), center + half_width


def estimator_histograms(concentration, variable, column):
    direct_hists = []
    direct_values = []
    direct_weights = []
    for replicate in direct_replicates[concentration]:
        values = replicate['state'][:, column]
        weights = replicate['weights']
        centers, density, _ = histogram_density(values, variable, weights)
        direct_hists.append(density)
        direct_values.append(values)
        direct_weights.append(weights / len(direct_replicates[concentration]))
    _, direct_pooled, direct_retained = histogram_density(
        np.concatenate(direct_values), variable, np.concatenate(direct_weights))

    trace_hists = []
    filtered_hists = []
    trace_values = []
    filtered_values = []
    for replicate in trace_replicates[concentration]:
        values = replicate['state'][:, column]
        filtered = values[replicate['keep_without_short_loops']]
        _, density, _ = histogram_density(values, variable)
        _, filtered_density, _ = histogram_density(filtered, variable)
        trace_hists.append(density)
        filtered_hists.append(filtered_density)
        trace_values.append(values)
        filtered_values.append(filtered)
    _, trace_pooled, trace_retained = histogram_density(
        np.concatenate(trace_values), variable)
    _, filtered_pooled, filtered_retained = histogram_density(
        np.concatenate(filtered_values), variable)
    return {
        'centers': centers,
        'direct': direct_pooled, 'direct_interval': replicate_interval(direct_hists),
        'trace': trace_pooled, 'trace_interval': replicate_interval(trace_hists),
        'filtered': filtered_pooled, 'filtered_interval': replicate_interval(filtered_hists),
        'retained': (direct_retained, trace_retained, filtered_retained),
    }


COLORS = {'direct': '#111111', 'trace': '#2166ac', 'filtered': '#b2182b'}
LINESTYLES = {'direct': '-', 'trace': '--', 'filtered': ':'}
LABELS = {
    'direct': 'GRF',
    'trace': 'trace',
    'filtered': r'trace, $Lk_{\mathrm{rms},0}>6\pi$',
}
panel_labels = tuple(f'({chr(ord("a") + index)})' for index in range(6))
fig, axes = plt.subplots(2, 3, figsize=(9.0, 6.0), squeeze=False)
figure_rows = []
for row_index, spectrum in enumerate(selected_spectra):
    concentration = spectrum.concentration_mM
    for column_index, (variable, state_column, xlabel, ylabel) in enumerate(VARIABLE_SPECS):
        ax = axes[row_index, column_index]
        curves = estimator_histograms(concentration, variable, state_column)
        for key in ('direct', 'trace', 'filtered'):
            ax.plot(curves['centers'], curves[key], color=COLORS[key],
                    linestyle=LINESTYLES[key], lw=1.5,
                    label=LABELS[key])
        dx = curves['centers'][1] - curves['centers'][0]
        l1_direct_trace = float(np.sum(np.abs(curves['direct'] - curves['trace'])) * dx)
        l1_trace_filtered = float(np.sum(np.abs(curves['trace'] - curves['filtered'])) * dx)
        figure_rows.append({
            'concentration_mM': concentration, 'variable': variable,
            'direct_retained_probability': curves['retained'][0],
            'trace_retained_probability': curves['retained'][1],
            'filtered_retained_probability': curves['retained'][2],
            'L1_direct_vs_trace': l1_direct_trace,
            'L1_trace_vs_filtered': l1_trace_filtered,
        })
        ax.set(xlabel=xlabel, ylabel=ylabel, xlim=HISTOGRAM_RANGES[variable],
               xticks=DISPLAY_TICKS[variable], yscale='log')
        ax.set_ylim(1e-4, 10.0)
        ax.set_box_aspect(1)
        ax.tick_params(axis='both', which='both', labelsize=10)
        ax.xaxis.label.set_size(14)
        ax.yaxis.label.set_size(14)
        ax.text(0.03, 0.97, panel_labels[row_index * 3 + column_index],
                transform=ax.transAxes, ha='left', va='top', fontsize=12,
                fontweight='bold')
        if column_index == 0:
            ax.text(0.97, 0.95, f'{concentration:g} mM', transform=ax.transAxes,
                    ha='right', va='top', fontsize=10)

handles, labels = axes[0, 2].get_legend_handles_labels()
axes[1, 0].legend(handles, labels, loc='lower left', frameon=False,
                  fontsize=8, handlelength=1.7, handletextpad=0.45)
fig.subplots_adjust(left=0.10, right=0.99, bottom=0.11, top=0.98, wspace=0.35, hspace=0.38)
figure_path = OUTPUT_DIR / 'test_stat_loop_marginals.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()
print('[saved]', figure_path)


In [ ]:
diagnostic_path = OUTPUT_DIR / 'test_stat_loop_trace_diagnostics.csv'
summary_path = OUTPUT_DIR / 'test_stat_loop_distribution_summary.csv'
def write_rows(path, rows):
    if not rows:
        raise ValueError(f'No rows available for {path}')
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

write_rows(diagnostic_path, trace_diagnostic_rows)
write_rows(summary_path, figure_rows)

# These checks deliberately fail loudly when a production run is not trustworthy.
if not QUICK_CHECK:
    if any(row['zero_surface_residual_rms'] > 1e-8 for row in trace_diagnostic_rows):
        raise RuntimeError('Newton projection residual is too large')
    if any(abs(row[f'spectral_moment_{order}_relative_error']) > 1e-11
           for row in trace_diagnostic_rows for order in (2, 4, 6)):
        raise RuntimeError('Traced fields do not reproduce the target spectral moments')
    for concentration in sorted({row['concentration_mM'] for row in trace_diagnostic_rows}):
        rejected = [row['quality_rejected_raw_arclength_fraction']
                    for row in trace_diagnostic_rows
                    if row['concentration_mM'] == concentration]
        if (np.mean(rejected) > MAX_MEAN_QUALITY_REJECTED_ARCLENGTH or
                np.max(rejected) > MAX_SINGLE_QUALITY_REJECTED_ARCLENGTH):
            raise RuntimeError(
                f'Excess rejected arclength at {concentration:g} mM: '
                f'mean={np.mean(rejected):.4f}, max={np.max(rejected):.4f}')
    if any(row['too_short_raw_arclength_fraction'] > 0.04
           for row in trace_diagnostic_rows):
        raise RuntimeError('More than 4% of raw traced arclength was too short to sample reliably')
    if any(row['topology_censored_arclength_fraction'] > 0.10
           for row in trace_diagnostic_rows):
        raise RuntimeError('More than 10% of retained arclength is topology-censored')
    if any(row['spacing_ratio_std'] > 0.15 for row in trace_diagnostic_rows):
        raise RuntimeError('Arclength sampling is not sufficiently uniform')

print('[saved]', diagnostic_path)
print('[saved]', summary_path)
print('trace diagnostics:')
for row in trace_diagnostic_rows:
    print(row)
print('distribution summary:')
for row in figure_rows:
    print(row)


## Tangent correlation and effective Kuhn length

For every salt concentration, the signed orientation correlation is calculated along each explicitly traced chain. The finite-box mean tangent is subtracted within each independent field realization and the connected result is normalized to $C_t(0)=1$; this removes numerical box polarization without discarding contours. All retained contours contribute, including short closed loops. Genuinely closed contours are correlated periodically only up to half their circumference; boundary- and topology-censored contours are never wrapped. Panel (b) follows the standard signed-integral definition, $b_K(s_{\max})=2\int_0^{s_{\max}}C_t(s)\,ds$, with the explicitly reported finite cutoff $s_{\max}k_{\mathrm{rms},0}=20$. The result is therefore a cutoff-dependent estimate rather than an asserted infinite-chain limit. The wormlike-chain exponential estimate and positive-lobe integral are retained in the output table as diagnostics.


In [ ]:
CORRELATION_MAX_QREF = 20.0
CORRELATION_MIN_PAIR_FRACTION = 0.02
EXPONENTIAL_FIT_CORRELATION_RANGE = (0.15, 0.80)
correlation_spectra = tuple(sorted(all_spectra, key=lambda row: row.concentration_mM))


def closure_aware_tangent_correlation(result, physical_spacing, max_separation):
    tangents_by_contour = result['contour_tangents']
    closed_by_contour = result['contour_closed']
    maximum_lag = int(np.floor(max_separation / physical_spacing + 1e-12))
    unit_tangents = []
    for tangent in tangents_by_contour:
        tangent = np.asarray(tangent, dtype=float)
        unit_tangents.append(tangent / np.linalg.norm(tangent, axis=1)[:, None])
    mean_tangent = np.mean(np.vstack(unit_tangents), axis=0)
    total = np.zeros(maximum_lag + 1, dtype=float)
    pair_count = np.zeros(maximum_lag + 1, dtype=np.int64)
    for tangent, closed in zip(unit_tangents, closed_by_contour):
        tangent = tangent - mean_tangent
        local_maximum = min(maximum_lag, len(tangent) // 2 if closed else len(tangent) - 1)
        for lag in range(local_maximum + 1):
            if closed:
                dots = np.einsum('ij,ij->i', tangent, np.roll(tangent, -lag, axis=0))
            elif lag == 0:
                dots = np.einsum('ij,ij->i', tangent, tangent)
            else:
                dots = np.einsum('ij,ij->i', tangent[:-lag], tangent[lag:])
            total[lag] += np.sum(dots)
            pair_count[lag] += len(dots)
    correlation = np.divide(total, pair_count, out=np.full_like(total, np.nan),
                            where=pair_count > 0)
    correlation /= correlation[0]
    keep = pair_count >= max(1, int(CORRELATION_MIN_PAIR_FRACTION * pair_count[0]))
    last = np.flatnonzero(keep)[-1]
    separation = np.arange(last + 1, dtype=float) * physical_spacing
    return separation, correlation[:last + 1], pair_count[:last + 1]


def positive_lobe_kuhn_length(separation, correlation):
    separation = np.asarray(separation, dtype=float)
    correlation = np.asarray(correlation, dtype=float)
    valid = np.isfinite(separation) & np.isfinite(correlation)
    separation = separation[valid]
    correlation = correlation[valid]
    crossing = np.flatnonzero(correlation[1:] <= 0)
    if len(crossing) == 0:
        return np.nan, np.nan
    upper = int(crossing[0] + 1)
    lower = upper - 1
    fraction = correlation[lower] / (correlation[lower] - correlation[upper])
    zero_separation = separation[lower] + fraction * (separation[upper] - separation[lower])
    integral_s = np.concatenate((separation[:upper], [zero_separation]))
    integral_c = np.concatenate((correlation[:upper], [0.0]))
    return 2.0 * float(np.trapezoid(integral_c, integral_s)), zero_separation


def signed_integral_kuhn_length(separation, correlation):
    separation = np.asarray(separation, dtype=float)
    correlation = np.asarray(correlation, dtype=float)
    valid = np.isfinite(separation) & np.isfinite(correlation)
    if np.count_nonzero(valid) < 2:
        return np.nan
    return 2.0 * float(np.trapezoid(correlation[valid], separation[valid]))


def exponential_effective_kuhn_length(separation, correlation):
    separation = np.asarray(separation, dtype=float)
    correlation = np.asarray(correlation, dtype=float)
    lower, upper = EXPONENTIAL_FIT_CORRELATION_RANGE
    keep = (np.isfinite(separation) & np.isfinite(correlation) &
            (correlation >= lower) & (correlation <= upper))
    if np.count_nonzero(keep) < 8:
        return np.nan, np.nan
    coordinate = separation[keep]
    log_correlation = np.log(correlation[keep])
    slope, intercept = np.polyfit(coordinate, log_correlation, 1)
    fitted = intercept + slope * coordinate
    residual = np.sum((log_correlation - fitted)**2)
    total = np.sum((log_correlation - np.mean(log_correlation))**2)
    fit_r2 = 1.0 - residual / total if total > 0 else np.nan
    return (-2.0 / slope if slope < 0 else np.nan), float(fit_r2)


correlation_results = {}
correlation_rows = []
kuhn_rows = []
for spectrum in correlation_spectra:
    concentration = spectrum.concentration_mM
    physical_spacing = TRACE_Q_SPACING / spectrum.k_eff
    replicate_curves = []
    replicate_pairs = []
    replicate_kuhn = []
    replicate_positive_lobe_kuhn = []
    replicate_exponential_kuhn = []
    replicate_zero = []
    replicate_fit_r2 = []
    for seed, result in zip(TRACE_SEEDS, trace_replicates[concentration]):
        separation, correlation, pair_count = closure_aware_tangent_correlation(
            result, physical_spacing, CORRELATION_MAX_QREF / K_EFF_0)
        replicate_curves.append(correlation)
        replicate_pairs.append(pair_count)
        kuhn_length = signed_integral_kuhn_length(separation, correlation)
        exponential_kuhn, fit_r2 = exponential_effective_kuhn_length(separation, correlation)
        positive_lobe_kuhn, zero_crossing = positive_lobe_kuhn_length(separation, correlation)
        replicate_kuhn.append(kuhn_length)
        replicate_exponential_kuhn.append(exponential_kuhn)
        replicate_fit_r2.append(fit_r2)
        replicate_positive_lobe_kuhn.append(positive_lobe_kuhn)
        replicate_zero.append(zero_crossing)
    common_count = min(map(len, replicate_curves))
    separation = separation[:common_count]
    curves = np.stack([curve[:common_count] for curve in replicate_curves])
    pairs = np.stack([count[:common_count] for count in replicate_pairs])
    mean_correlation = np.mean(curves, axis=0)
    correlation_sem = np.std(curves, axis=0, ddof=1) / np.sqrt(len(curves))
    kuhn_values = np.asarray(replicate_kuhn, dtype=float)
    finite_kuhn = kuhn_values[np.isfinite(kuhn_values)]
    kuhn_mean = float(np.mean(finite_kuhn))
    kuhn_sem = float(np.std(finite_kuhn, ddof=1) / np.sqrt(len(finite_kuhn)))
    critical = student_t.ppf(0.975, df=len(finite_kuhn) - 1)
    correlation_results[concentration] = {
        'separation_A': separation, 'correlation': mean_correlation,
        'correlation_sem': correlation_sem, 'pair_count': np.sum(pairs, axis=0),
        'kuhn_length_A': kuhn_mean, 'kuhn_ci95_A': critical * kuhn_sem,
    }
    for s, value, sem, count in zip(separation, mean_correlation, correlation_sem,
                                     np.sum(pairs, axis=0)):
        correlation_rows.append({
            'concentration_mM': concentration, 'separation_A': s,
            'separation_k_eff_0': s * K_EFF_0, 'tangent_correlation': value,
            'replicate_standard_error': sem, 'pair_count': int(count),
        })
    kuhn_rows.append({
        'concentration_mM': concentration, 'replicate_count': len(finite_kuhn),
        'integration_max_separation_k_eff_0': separation[-1] * K_EFF_0,
        'signed_integral_kuhn_length_A': kuhn_mean,
        'signed_integral_kuhn_length_k_eff_0': kuhn_mean * K_EFF_0,
        'signed_integral_kuhn_ci95_A': critical * kuhn_sem,
        'signed_integral_kuhn_ci95_k_eff_0': critical * kuhn_sem * K_EFF_0,
        'exponential_effective_kuhn_A': float(np.nanmean(replicate_exponential_kuhn)),
        'mean_exponential_fit_r2': float(np.nanmean(replicate_fit_r2)),
        'positive_lobe_integral_kuhn_A': float(np.nanmean(replicate_positive_lobe_kuhn))
            if np.any(np.isfinite(replicate_positive_lobe_kuhn)) else np.nan,
        'mean_first_zero_A': float(np.nanmean(replicate_zero))
            if np.any(np.isfinite(replicate_zero)) else np.nan,
    })

salt_colors = plt.cm.rainbow(np.linspace(0.0, 1.0, len(correlation_spectra)))
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.6))
for spectrum, color in zip(correlation_spectra, salt_colors):
    concentration = spectrum.concentration_mM
    result = correlation_results[concentration]
    separation_qref = result['separation_A'] * K_EFF_0
    correlation = result['correlation']
    axes[0].plot(separation_qref[1:], correlation[1:],
                 color=color, lw=1.5, label=f'{concentration:g} mM')
axes[0].set(xlabel=r'$s k_{\mathrm{rms},0}$', ylabel=r'$C_t(s)$',
            yscale='log', xlim=(0.0, CORRELATION_MAX_QREF),
            ylim=(1e-3, 1.05))
axes[0].legend(ncol=2, frameon=False, fontsize=7, loc='upper right',
               columnspacing=0.8, handlelength=1.5, handletextpad=0.4)
for row, color in zip(kuhn_rows, salt_colors):
    axes[1].errorbar(row['concentration_mM'],
                     row['signed_integral_kuhn_length_k_eff_0'],
                     yerr=row['signed_integral_kuhn_ci95_k_eff_0'],
                     fmt='o', color=color,
                     markeredgecolor=color, markerfacecolor='none', capsize=2.5, ms=5)
axes[1].set(xlabel=r'CaCl$_2$ (mM)',
            ylabel=r'$b_K k_{\mathrm{rms},0}$')
for panel_index, ax in enumerate(axes):
    ax.set_box_aspect(1)
    ax.tick_params(axis='both', which='both', labelsize=10)
    ax.xaxis.label.set_size(14)
    ax.yaxis.label.set_size(14)
    ax.text(0.03, 0.97, f'({chr(ord("a") + panel_index)})', transform=ax.transAxes,
            ha='left', va='top', fontsize=12, fontweight='bold')
fig.subplots_adjust(left=0.10, right=0.99, bottom=0.17, top=0.97, wspace=0.34)
correlation_figure_path = OUTPUT_DIR / 'test_stat_loop_tangent_correlation.png'
fig.savefig(correlation_figure_path, dpi=300, bbox_inches='tight')
plt.show()

correlation_data_path = OUTPUT_DIR / 'test_stat_loop_tangent_correlation.csv'
kuhn_data_path = OUTPUT_DIR / 'test_stat_loop_effective_kuhn_length.csv'
write_rows(correlation_data_path, correlation_rows)
write_rows(kuhn_data_path, kuhn_rows)
print('[saved]', correlation_figure_path)
print('[saved]', correlation_data_path)
print('[saved]', kuhn_data_path)
kuhn_rows


## Reading the comparison

The black-to-blue difference measures the combined finite-grid, finite-window, and tracing error. The blue-to-red difference isolates the contribution of short **closed** loops under the chosen cutoff. The latter should not be interpreted by removing short boundary-censored traces: their complete loop length is unknown. Before drawing a physical conclusion, increase the grid size, number of modes, and number of field seeds until the blue curve and the short-loop fraction are stable within their reported intervals, and repeat the calculation for at least two values of `TRACE_Q_SPACING`.
